In [ ]:
import subprocess
import time
import requests
from bs4 import BeautifulSoup
import random
import mysql.connector
import pandas as pd

In [ ]:
#Connection to my SQL server, parameters must be filled with correct information otherwise it won't work
conn = mysql.connector.connect(
    host = 'localhost',
    user = 'root',
    password = 'dukagro',
    database = 'test'
)
cursor = conn.cursor()

In [ ]:
#Representing the webpage url, where only u will be changing in iterations
BASE_URL = "https://bitcointalk.org/index.php?action=profile;u={}" 

HEADERS_LIST = [
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/118.0.5993.118 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    },
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:118.0) "
                      "Gecko/20100101 Firefox/118.0",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    },
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Edge/118.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    },
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; Trident/7.0; rv:11.0) like Gecko",
        "Accept": "text/html, application/xhtml+xml, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    },
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/112.0.5615.138 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    }
]

In [ ]:
def extract_field(soup, field_name):
    """
    Extracts a specific field from the profile, e.g., Location or Bitcoin address.
    If the field exists but is empty, returns None.
    If the field does not exist, also returns None.
    """
    # Find all table rows in the parsed HTML
    rows = soup.find_all("tr")
    for row in rows:
        # Find all data cells (<td>) within the current row
        cells = row.find_all("td")

        # We expect exactly two cells for a valid Label:Value pair
        if len(cells) == 2:
            # Extract text from the first cell, strip whitespace, and remove colons
            label = cells[0].get_text(strip=True).replace(":", "")
            # Check if the extracted label matches the target field name (case-insensitive)
            if label.lower() == field_name.lower():
                value = cells[1].get_text(strip=True)
                return value if value else None
    return None


def get_profile_info(user_id, res):
    """
    Parses the HTTP response of a user profile to extract their Bitcoin address and Location.
    """
    # Parse the raw HTML content from the response
    soup = BeautifulSoup(res.text, "html.parser")

    # Attempt to extract the 'Location' and 'Bitcoin address' fields
    location = extract_field(soup, "Location")
    bitcoin_address = extract_field(soup, "Bitcoin address")

    # If the user has not provided a Bitcoin address, we skip saving the location
    if bitcoin_address == None:
        #print(f"User ID: {user_id}")
        row = [user_id, None, None]
        return row

    location = location if location else None
    bitcoin_address = bitcoin_address if bitcoin_address else None
    print(f"User ID: {user_id}")
    print(f"  Location: {location}")
    print(f"  Bitcoin Address: {bitcoin_address}")
    print("-" * 50)
    row = [user_id, bitcoin_address, location]
    return row

# Define the file path for the proxy list
path = r"Webshare 10 proxies.txt"
proxies_dict_list = []

# Open the text file containing the proxies in read mode with UTF-8 encoding
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        # split the TXT format IP:PORT:USER:PASS
        ip, port, user, pwd = line.split(":")
        
        # create a dict directly for requests
        proxy_dict = {
            "http": f"http://{user}:{pwd}@{ip}:{port}",
            "https": f"http://{user}:{pwd}@{ip}:{port}"
        }

        proxies_dict_list.append(proxy_dict)

In [ ]:
#Setting start_id and end_id because scraping was done in batches
time1 = time.time()
start_id = 0
end_id = 100000
i = 0

data = []
numbers = list(range(start_id, end_id +1))

for uid in numbers:
    url = BASE_URL.format(uid)
    #Trying to get data from one url for three times, each time the random proxy (IP address from the proxy_dict) is chosen and random header
    try:
        res = requests.get(url, proxies = random.choice(proxies_dict_list), headers=random.choice(HEADERS_LIST), timeout = 10)
    except requests.exceptions.RequestException:
        print("error")
        time.sleep(5)
        try:
            res = requests.get(url, proxies = random.choice(proxies_dict_list), headers=random.choice(HEADERS_LIST), timeout = 10)
        except requests.exceptions.RequestException:
            print("error")
            time.sleep(5)
            res = requests.get(url, proxies = random.choice(proxies_dict_list), headers=random.choice(HEADERS_LIST), timeout = 10)
    
    if res.status_code != 200:
        print(f"User {uid}: not reachable")
        res = requests.get(url, proxies = random.choice(proxies_dict_list), headers=random.choice(HEADERS_LIST), timeout = 10)

    #Getting information from user
    row = get_profile_info(uid, res)
    data.append(row)
    i += 1

    #Once in every 1000 scraped users the whole block of data is written into mysql database
    if i%1000 == 0:
        cursor.executemany("""INSERT IGNORE INTO scraped_data (id, address, location) VALUES (%s, %s, %s)""", data)
        conn.commit()
        data = []
        print(time.time()-time1)